# 🍇 ModelVinya · Etiquetatge automàtic amb SAM 3

Aquest notebook fa servir **SAM 3** (Meta) per segmentar automàticament les baies d'un racim i genera un **projecte `.json` de ModelVinya** (cercles + diàmetres equivalents) que pots obrir directament a l'anotador per **revisar i corregir**.

## Abans de començar
1. **Runtime amb GPU:** menú *Entorn d'execució → Canviar el tipus d'entorn → GPU*.
2. **Accés als checkpoints (gated):** demana accés a https://huggingface.co/facebook/sam3 i crea un token a https://huggingface.co/settings/tokens
3. Executa les cel·les **en ordre** (de dalt a baix).

> El prompt de text per defecte és `berry`. En racims molt densos SAM 3 pot segmentar grups; ajusta el `PROMPT` i els llindars d'àrea més avall, i acaba de polir a l'anotador. SAM 3 no compta baies amagades: això ho corregeix després el model d'oclusió (fase 2).

In [ ]:
# 1) Instal·lar SAM 3
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip -q install -e .
!pip -q install huggingface_hub pillow numpy

In [ ]:
# 2) Accés als checkpoints (gated). Enganxa el teu token de Hugging Face quan ho demani.
from huggingface_hub import login
login()   # alternativa: !hf auth login

In [ ]:
# 3) Carregar el model SAM 3 (descarrega el checkpoint facebook/sam3 la primera vegada)
import glob
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# La instal.lacio editable (-e) de vegades no localitza el tokenizer BPE i passa None
# -> el busquem dins del repo clonat i el passem explicitament a build_sam3_image_model.
_bpe = glob.glob('/content/sam3/**/bpe_simple_vocab_16e6.txt.gz', recursive=True)
bpe_path = _bpe[0] if _bpe else None
print('Tokenizer BPE:', bpe_path)

model = build_sam3_image_model(bpe_path=bpe_path)
processor = Sam3Processor(model)
print('SAM 3 carregat \u2714')

In [ ]:
# 4) Paràmetres i pujada d'imatges
PROMPT        = 'grape'    # concepte a segmentar ('grape'/'grapes' van millor que 'berry')
SCORE_MIN     = 0.5        # confiança mínima per acceptar una detecció
AREA_MIN_FRAC = 0.00002    # àrea mínima de baia respecte la imatge (descarta soroll)
AREA_MAX_FRAC = 0.05       # àrea màxima (descarta el racim sencer / el fons)

from google.colab import files
uploaded = files.upload()           # selecciona una o més fotos de racims
image_paths = list(uploaded.keys())
print('Imatges:', image_paths)

In [ ]:
# 5) Inferencia SAM 3  ->  cercles  ->  projecte ModelVinya (.json)
import numpy as np, base64, json, mimetypes, torch
from datetime import datetime, timezone
from PIL import Image

def to_np(x):
    """Tensor de PyTorch (GPU/bfloat16) o array -> numpy float32 a CPU."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().float()
    return np.asarray(x)

def to_binary(mask):
    arr = np.squeeze(to_np(mask))
    if arr.dtype == bool:
        return arr
    return arr > 0.5 if arr.max() <= 1.0 else arr > 0

def mask_to_circle(binary):
    ys, xs = np.nonzero(binary)
    if len(xs) == 0:
        return None
    area = float(len(xs))
    # Diametre equivalent (PDF):  D = 2*sqrt(A/pi)  ->  r = sqrt(A/pi)
    return {"cx": round(float(xs.mean()), 2),
            "cy": round(float(ys.mean()), 2),
            "r":  round(float(np.sqrt(area / np.pi)), 2),
            "area_px": area}

def data_url(path):
    mime = mimetypes.guess_type(path)[0] or 'image/jpeg'
    with open(path, 'rb') as f:
        return f"data:{mime};base64," + base64.b64encode(f.read()).decode('ascii')

EMPTY_FICHA  = {"id_racimo": "", "fecha": "", "variedad": "", "fase_fenologica": "",
                "tratamiento": "", "vigor": "", "orientacion": "", "sistema_conduccion": ""}
EMPTY_VERDAD = {"N_total_real": "", "Diam_real_mm": "", "Peso_baya_real_g": "", "Peso_racimo_real_g": ""}

# Deixem passar candidats (filtrem despres per SCORE_MIN i area)
processor.set_confidence_threshold(0.05)

images_out = []
for path in image_paths:
    image = Image.open(path).convert('RGB')
    W, H = image.size
    with torch.autocast('cuda', dtype=torch.bfloat16):   # el model corre en bfloat16
        state  = processor.set_image(image)
        output = processor.set_text_prompt(state=state, prompt=PROMPT)

    masks  = to_np(output["masks"])           # (N, H, W) binaries
    scores = to_np(output["scores"]).reshape(-1)
    print(f"{path}: SAM 3 ha retornat {len(masks)} mascares")

    amin, amax = AREA_MIN_FRAC * W * H, AREA_MAX_FRAC * W * H
    bayas = []
    for i in range(len(masks)):
        if i < len(scores) and scores[i] < SCORE_MIN:
            continue
        c = mask_to_circle(to_binary(masks[i]))
        if c is None or c["area_px"] < amin or c["area_px"] > amax:
            continue
        bayas.append({"cx": c["cx"], "cy": c["cy"], "r": c["r"]})

    images_out.append({
        "nombre": path, "dataURL": data_url(path), "w": W, "h": H,
        "escala_mm_px": None, "escalaLinea": None,
        "bayas": bayas, "racimo": None,
        "ficha": dict(EMPTY_FICHA), "verdad": dict(EMPTY_VERDAD),
    })
    print(f"  -> {len(bayas)} baies acceptades (filtrades per score i area)")

project = {"app": "ModelVinya", "version": 1,
           "savedAt": datetime.now(timezone.utc).isoformat(),
           "images": images_out}

OUT = "modelvinya_sam3_proyecto.json"
with open(OUT, "w") as f:
    json.dump(project, f)
print("\nGuardat:", OUT, "|", sum(len(im["bayas"]) for im in images_out), "baies en total")

In [ ]:
# 6) Descarregar el projecte generat
from google.colab import files
files.download('modelvinya_sam3_proyecto.json')

## Com fer-ho servir a l'anotador
1. Obre l'anotador: **https://planessoria-ui.github.io/ModelVinya/**
2. **📦 Abrir proyecto** → tria `modelvinya_sam3_proyecto.json`.
3. Veuràs les baies **pre-marcades amb cercles**. Revisa i corregeix: mou / redimensiona, esborra falsos positius (Supr) i afegeix els que falten (⭕).
4. **Defineix l'escala** (📏) sobre la referència real per tenir els diàmetres en mm.
5. Omple la fitxa i la veritat de camp, i exporta **CSV / YOLO** com sempre.

### Si la detecció no és bona
- **Racims densos:** prova `PROMPT = 'grape'` o `'round green berry'`, o baixa `AREA_MAX_FRAC`.
- **Massa falsos positius:** puja `SCORE_MIN`.
- **Es perden baies petites:** baixa `AREA_MIN_FRAC`.
- SAM 3 també accepta **punts/caixes** com a prompt visual (exemplars); es pot ampliar el notebook si el text sol no n'hi ha prou.